# Agent Observability & Failure-Mode Handling

## Why this matters

This is the topic that the X++ project's entire debugging arc was actually about, without it being named as "observability" at the time. Every session that started with "why are the findings varying" or "why did the JSON break" was an observability problem first -- something failed or behaved unexpectedly, and the real work was figuring out *why* before it could be fixed. `AgentCallError` raising instead of silently returning `[]`, the `stop_reason` checks, the synthetic test files built to catch bugs before shipping -- all of that *is* agent observability and failure-mode handling, just built incrementally as specific problems came up rather than designed upfront as a coherent system. This notebook names the general patterns behind what was already done, plus the ones that weren't needed yet but would be for a more complex agent (loop guards, retry logic, tracing).

## Level 1: What it is

**Observability**: the ability to understand *why* an agent produced a given output (or failed to), after the fact, without having to guess. **Failure-mode handling**: the design patterns that keep a single failure from silently corrupting output or spiraling into worse failures (infinite loops, cascading errors, runaway cost). Four categories worth distinguishing:

1. **Logging/tracing** -- recording what actually happened (which tool was called, what arguments, what came back) so it can be inspected later
2. **Explicit failure signaling** -- making failures visible and distinguishable from legitimate "nothing found" results, rather than letting them look identical
3. **Loop/runaway guards** -- preventing an agent (especially one with a supervisor/tool-calling loop, per the multi-agent notebook) from calling tools indefinitely without making progress
4. **Retry and backoff patterns** -- handling transient failures (rate limits, network blips) without either giving up too early or hammering a failing endpoint

## Level 2: How it actually works internally

### Explicit failure signaling: what `base_agent.py` already does, named properly

The core principle: **an empty/default result must never be ambiguous between "nothing to report" and "something went wrong."** This is exactly the reasoning documented directly in `base_agent.py`'s own module docstring: the original free-text-JSON version's failure mode was that a truncated or malformed response would fail to parse and *silently return `[]`* -- which is indistinguishable from a genuinely clean security review. The fix wasn't just better parsing, it was a design principle: `AgentCallError` gets raised on any unrecoverable failure, so the caller (`app.py`'s per-agent `try/except`) can show "Security failed: ..." rather than a false-clean "0 issues found." This is the single most important observability pattern in the whole project, and it generalizes well beyond this one system: **any function that can legitimately return "nothing" also needs a way to signal "I don't know" that's distinguishable from that legitimate nothing.**

### Tracing: making the invisible decision path visible

For a single `messages.create()` call, there isn't much to trace beyond input/output -- but once a system involves a supervisor loop (per the multi-agent notebook) or multiple sequential tool calls, tracing means recording *each* step: which tool was called, with what arguments, what it returned, and what the model did with that result next. Without this, debugging a multi-step agent means trying to reconstruct what happened from the final output alone -- often impossible, since the same final output could have been reached via very different paths. Production agent systems typically log every tool call and its result as a structured trace (not just the final answer), specifically so a failure three steps into a five-step process can be pinpointed to that exact step rather than requiring the whole interaction to be treated as one opaque black box.

### Loop/runaway guards

A supervisor pattern (from the multi-agent notebook) that decides at runtime which tools to call, and can call them repeatedly based on results, has no inherent stopping point unless one is built in. The standard guard: a hard cap on iterations ("if the loop hasn't converged after N tool calls, stop and surface what's been gathered so far rather than continuing indefinitely") -- this matters for both cost (an unbounded loop means unbounded API spend) and correctness (a model that keeps calling the same tool with slightly different arguments, hoping for a different result, is a real observed failure pattern, not a hypothetical). The X++ project's fixed parallel fan-out never needed this, precisely because it isn't a loop at all -- every agent runs exactly once, which is part of why that pattern is simpler to reason about than a supervisor loop would be.

### Retry and backoff

Transient failures (rate limits, momentary network issues) are different from the deterministic failures `base_agent.py` handles (a truncated response, a malformed tool call) -- a transient failure might succeed if simply retried, where a truncation failure will reliably fail again with the exact same input. The standard pattern is **exponential backoff**: retry after a short delay, and if it fails again, wait longer before the next retry (doubling or similar), up to a capped maximum number of attempts -- this avoids hammering an already-struggling endpoint with immediate retries while still recovering automatically from brief blips. Retrying `AgentCallError`s from truncation or malformed output would NOT help, since re-sending the identical input to a `temperature=0` call tends to reproduce the same failure -- retry logic needs to distinguish transient/infrastructure failures (worth retrying) from deterministic/logical failures (retrying won't fix a schema mismatch).

## Level 3: Where it breaks, and what it doesn't solve

1. **Observability tells you *that* something failed and *where* -- it doesn't tell you *why the model made the judgment it did*.** All the tracing in the world shows you the Security agent's authorization-check finding was present in 2 of 3 runs, but doesn't explain the model's internal reasoning for including or omitting it -- that's fundamentally opaque in the way the LLM-as-judge notebook discussed (a second model opinion narrows the mystery, it doesn't eliminate it). Observability and interpretability are different problems; good logging solves the first, not the second.

2. **Loop guards trade completeness for safety, and that trade needs to be a conscious choice, not an accident.** A hard iteration cap means a genuinely complex task that needs, say, 12 tool calls to fully resolve gets cut off at a cap of 10 and returns a partial result -- which is safer than an infinite loop, but is itself a form of the same problem `base_agent.py` was built to avoid: a partial/incomplete result needs to be clearly flagged as partial, not presented as if it were complete.

3. **Blind retries can turn a cheap failure into an expensive one.** Retrying a call that fails due to a genuinely oversized input (e.g., a class so large it will always exceed `max_tokens` no matter how many times you retry) just multiplies the cost of an inevitable failure by however many retry attempts are configured, without ever succeeding -- retry logic needs is a genuine judgment about which failure categories are worth retrying at all, not blanket retry-on-any-exception.

4. **More logging is not automatically more useful logging.** A trace of every single tool call and token in a long agent run can itself become too voluminous to actually read when something goes wrong -- structured, filterable logs (what failed, at which step, with what inputs) are more useful for actual debugging than raw verbose transcripts, same principle as the X++ HTML report's filter bar being more useful than an undifferentiated list of 40 issues.

5. **None of this replaces the ground-truth problem from the evaluation notebook.** Observability lets you see *that* the Security agent's output changed between runs and trace *what* it looked like each time -- it doesn't tell you *which* run was actually correct. Good observability is necessary for debugging reproducibility issues (you can't fix what you can't see), but it's a different problem from evaluation/correctness, covered in the LLM-as-judge notebook -- conflating "I can see what happened" with "I know if it was right" is a common mistake.

In [ ]:
# Minimal exponential backoff wrapper -- the retry pattern discussed
# above, applied correctly: only retrying failure categories that are
# plausibly transient, NOT retrying AgentCallError from truncation/
# malformed output (since that would just waste calls reproducing the
# same deterministic failure at temperature=0).

import time
from anthropic import APIStatusError, APIConnectionError


def call_with_backoff(fn, *args, max_attempts=3, base_delay=1.0, **kwargs):
    """Retry fn only on plausibly-transient errors (rate limits, connection
    issues) -- NOT on AgentCallError from truncation or malformed tool
    output, since retrying an identical temperature=0 call that failed
    deterministically will just fail again the same way, at extra cost."""
    for attempt in range(max_attempts):
        try:
            return fn(*args, **kwargs)
        except (APIStatusError, APIConnectionError) as e:
            if attempt == max_attempts - 1:
                raise
            delay = base_delay * (2 ** attempt)
            print(f"Transient error ({e}), retrying in {delay}s "
                  f"(attempt {attempt + 1}/{max_attempts})")
            time.sleep(delay)
        # Deliberately NOT catching AgentCallError here -- it propagates
        # immediately, same as it does today in base_agent.py/app.py,
        # because retrying it wouldn't help and would hide the real signal.


print("call_with_backoff wraps only transient-failure categories -- "
      "AgentCallError still propagates on first occurrence, matching "
      "the 'never silently hide a real failure' principle already "
      "built into base_agent.py.")

## Interview-ready summary

**Q: "How do you make an LLM-based system debuggable and safe to run unattended?"**

Weak answer: "I'd add logging and a try/except around the API calls."

Strong answer: "The core principle is that any result which can legitimately be 'nothing found' needs an explicit, distinguishable way to signal 'something went wrong' instead -- an empty result and a failed call must never look the same to the caller. I built this into a code review agent by raising an explicit error on truncation or malformed output rather than returning an empty list, which turned a dangerous silent failure -- a security review reporting zero issues because it actually broke, not because the code was clean -- into a visible one. Beyond that, for anything involving a tool-calling loop rather than a single fixed call, you need iteration caps to prevent runaway cost or infinite loops, and retry logic that distinguishes transient failures worth retrying from deterministic failures that will just fail identically again. And observability isn't the same problem as correctness -- good tracing tells you what happened and where, but it doesn't tell you whether the model's judgment was right, which is a separate evaluation problem."